# French BESS stacked-services market walkthrough

This notebook is a working example of Feature 010. Its default path is fully offline: it loads the validated France 2022-08-19 market and activation artifacts, reproduces the deterministic battery replay, and explains the variables used to stack day-ahead energy arbitrage with symmetric Frequency Containment Reserve (FCR). A separate section demonstrates explicit live acquisition only when deliberately enabled.

The example uses one 100 kWh / 50 kW battery, hourly day-ahead decisions, four-hour FCR commitments, 15-minute state checkpoints, and 10-second physical activation. Positive battery power means **charging**; positive activation means an upward/discharge FCR request, so the activation contribution enters battery power with the opposite sign.


## What this feature demonstrates

The walkthrough covers the complete offline consumer path:

1. **Canonical market inputs** — validated ENTSO-E R3 A44 day-ahead prices and combined A81 FCR volume/capacity-price blocks.
2. **Separate activation input** — a provenanced 10-second RTE frequency-derived normalized command in `[-1, 1]`.
3. **Deterministic control** — fixed FCR capacity plus a 475/540 EUR/MWh price-threshold energy baseline clipped to preserve symmetric reserve headroom.
4. **Multi-rate physical replay** — requested and delivered combined/FCR power, state of charge, stored energy, throughput, tracking, non-delivery, and saturation.
5. **Settlement and audit** — day-ahead and FCR capacity revenue, explicit penalty status, terminal-SOC reporting, immutable input/schedule/trace/result fingerprints, and independently recorded totals.

The default notebook run does not fetch live data, read `.env`, optimize bids, or modify fixtures. The optional live section is disabled by default and reaches the existing explicit producer only after the user changes its guard. Acquisition and refresh are documented in `docs/architecture/entsoe-transparency-platform.md`.


## Restricted-data handling

> **Private-repository data:** the three `france_2022-08-19_*` fixtures and any executed cell output or exported figure containing their values are restricted to authorized collaborators. Do not publish the executed notebook, screenshots, HTML exports, figures, public CI artifacts, forks, mirrors, packages, or release attachments. Remove the restricted fixtures before making the repository public.

This source notebook is committed with empty outputs. Optional figure export writes only beneath ignored `tmp/`; keep `EXPORT_FIGURES = False` unless a private local export is needed. See `docs/migration/restricted-reference-fixtures.md`.


## 0. Imports and plotting setup

The core replay remains in Open Energy Lab. Pandas, Matplotlib, Seaborn, and IPython are notebook-only analysis dependencies installed through `uv sync --group notebook`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from examples.bess_stacked_services.run_offline_replay import run_offline_replay
from open_energy_lab.market import (
    load_market_data_artifact,
    load_regulation_activation_trace,
)


In [ ]:
sns.set_theme(context="notebook", style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160})

def find_repo_root(start=Path.cwd()):
    """Find the repository whether the kernel starts at root or below it."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "tests/fixtures/market"
        ).is_dir():
            return candidate
    raise RuntimeError("run this notebook from inside the Open Energy Lab repository")

REPO_ROOT = find_repo_root()
FIXTURE_DIR = REPO_ROOT / "tests/fixtures/market"
FIGURE_DIR = REPO_ROOT / "tmp/bess-stacked-services-notebook/figures"
EXPORT_FIGURES = False
LOCAL_TIMEZONE = "Europe/Paris"

def finish_figure(fig, filename):
    """Display one figure and optionally save it under ignored tmp/."""
    fig.tight_layout()
    if EXPORT_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        output = FIGURE_DIR / filename
        fig.savefig(output, bbox_inches="tight")
        print("Restricted local figure saved to", output)
    plt.show()


## 1. Load, validate, replay, and reconcile

`run_offline_replay` validates all three artifacts, rebuilds the recorded hourly and four-hour schedules, runs the 10-second physical simulation, settles the services, and rejects any mismatch in expected totals or fingerprints. Loading the market and activation artifacts separately below is for inspection only; it performs no network access.


In [ ]:
result, expected = run_offline_replay(FIXTURE_DIR)
market = load_market_data_artifact(
    FIXTURE_DIR / "france_2022-08-19_market_v1.json"
)
activation = load_regulation_activation_trace(
    FIXTURE_DIR / "france_2022-08-19_activation_v1.json"
)

policy = market.retention_and_license["restricted_repository_policy"]
assert market.retention_and_license["repository_storage"] == "permitted_private_restricted"
assert market.retention_and_license["repository_publication"] == "prohibited"
assert policy["package_distribution"] == "prohibited"
assert result.validation_status == "validated"

artifact_summary = pd.DataFrame(
    [
        {
            "artifact": "market",
            "schema": market.schema_version,
            "resolution/count": "24 hourly + 6 four-hour blocks",
            "artifact_fingerprint": market.artifact_fingerprint,
        },
        {
            "artifact": "activation",
            "schema": activation.schema_version,
            "resolution/count": f"{activation.resolution} / {len(activation.samples):,}",
            "artifact_fingerprint": activation.artifact_fingerprint,
        },
        {
            "artifact": "replay result",
            "schema": expected["schema_version"],
            "resolution/count": f"PT10S / {len(result.samples):,}",
            "artifact_fingerprint": result.result_fingerprint,
        },
    ]
)
display(artifact_summary)


## 2. Build analysis tables without losing simulation detail

The tables retain every canonical market interval and every 10-second replay sample. Plotting uses the local Europe/Paris timestamp for interpretation while preserving UTC columns for reproducibility. `sample_frame` includes baseline and FCR requests, combined and FCR delivery, tracking errors, non-delivery, charge/discharge energy, stored energy, SOC, and saturation causes.


In [ ]:
def add_local_time(frame):
    frame = frame.copy()
    frame["start_utc"] = pd.to_datetime(frame["start_utc"], utc=True)
    frame["end_utc"] = pd.to_datetime(frame["end_utc"], utc=True)
    frame["start_local"] = frame["start_utc"].dt.tz_convert(LOCAL_TIMEZONE)
    frame["end_local"] = frame["end_utc"].dt.tz_convert(LOCAL_TIMEZONE)
    return frame

def market_frame(artifact, series_id, value_name):
    series = artifact.series[series_id]
    return add_local_time(
        pd.DataFrame(
            [
                {
                    "start_utc": item.interval.start_utc,
                    "end_utc": item.interval.end_utc,
                    value_name: item.value,
                    "document_ref": item.document_ref,
                    "source_position": item.source_position,
                    "conversion_ref": item.conversion_ref,
                }
                for item in series.observations
            ]
        )
    )

def schedule_frame(records):
    return add_local_time(pd.DataFrame(records))

def step_data(frame, value_name):
    """Extend interval values through the final recorded end boundary."""
    x = frame["start_local"].tolist() + [frame["end_local"].iloc[-1]]
    y = frame[value_name].tolist() + [frame[value_name].iloc[-1]]
    return x, y

def nested_interval_frame(records):
    rows = []
    for record in records:
        row = dict(record)
        interval = row.pop("interval")
        row.update(start_utc=interval["start_utc"], end_utc=interval["end_utc"])
        rows.append(row)
    return add_local_time(pd.DataFrame(rows))

day_ahead = market_frame(market, "day_ahead_price", "price_eur_per_mwh")
fcr_volume = market_frame(market, "fcr_contracted_volume", "contracted_volume_mw")
fcr_price = market_frame(market, "fcr_capacity_price", "capacity_price_eur_per_mw_h")
energy_schedule = schedule_frame(expected["energy_schedule"]["intervals"])
fcr_schedule = schedule_frame(expected["fcr_schedule"]["intervals"])
decisions = schedule_frame(expected["baseline_derivation"]["intervals"])
decisions["headroom_lower_kw"] = decisions["admissible_power_interval_kw"].map(lambda value: value[0])
decisions["headroom_upper_kw"] = decisions["admissible_power_interval_kw"].map(lambda value: value[1])

sample_rows = []
for sample in result.samples:
    row = sample.to_dict()
    interval = row.pop("interval")
    row["start_utc"] = pd.Timestamp(interval["start_utc"])
    row["end_utc"] = pd.Timestamp(interval["end_utc"])
    row["saturation_causes"] = " | ".join(row["saturation_causes"]) or "none"
    sample_rows.append(row)
sample_frame = add_local_time(pd.DataFrame(sample_rows)).sort_values("start_utc")

checkpoint_frame = pd.DataFrame(item.to_dict() for item in result.state_checkpoints)
checkpoint_frame["timestamp_utc"] = pd.to_datetime(checkpoint_frame["timestamp_utc"], utc=True)
checkpoint_frame["timestamp_local"] = checkpoint_frame["timestamp_utc"].dt.tz_convert(LOCAL_TIMEZONE)
energy_summary = nested_interval_frame(item.to_dict() for item in result.energy_summaries)
settlement_payload = result.settlement.to_dict()
energy_settlement = nested_interval_frame(settlement_payload["energy_intervals"])
fcr_settlement = nested_interval_frame(settlement_payload["fcr_blocks"])

table_inventory = pd.DataFrame(
    {
        "table": ["day_ahead", "fcr_volume", "fcr_price", "decisions", "sample_frame", "checkpoint_frame"],
        "rows": [len(day_ahead), len(fcr_volume), len(fcr_price), len(decisions), len(sample_frame), len(checkpoint_frame)],
        "resolution": ["PT1H", "PT4H", "PT4H", "PT1H", "PT10S", "PT15M"],
    }
)
display(table_inventory)

variable_catalog = pd.DataFrame(
    [
        ("market", "price_eur_per_mwh", "EUR/MWh", "Hourly A44 day-ahead settlement price"),
        ("market", "contracted_volume_mw", "MW", "Total symmetric FCR market volume"),
        ("market", "capacity_price_eur_per_mw_h", "EUR/MW/h", "FCR capacity remuneration"),
        ("control", "unmodified_threshold_request_kw", "kW", "Price-threshold controller request before headroom clipping"),
        ("control", "headroom_lower_kw / headroom_upper_kw", "kW", "Admissible baseline range after symmetric FCR reservation"),
        ("control", "fixed_baseline_power_kw", "kW", "Accepted hourly energy schedule; positive means charging"),
        ("control", "fcr_capacity_kw", "kW", "Accepted symmetric reserve commitment"),
        ("physical", "activation", "[-1, 1]", "Normalized upward/discharge-positive FCR command"),
        ("physical", "requested_fcr_power_kw / delivered_fcr_power_kw", "kW", "Requested and delivered response relative to the baseline"),
        ("physical", "requested_combined_power_kw / delivered_combined_power_kw", "kW", "Total requested and feasible battery-terminal power"),
        ("battery", "charge_energy_kwh / discharge_energy_kwh", "kWh", "Terminal-side interval energy in each direction"),
        ("battery", "stored_energy_before_kwh / stored_energy_after_kwh", "kWh", "Efficiency-aware battery energy state"),
        ("battery", "soc_before / soc_after", "fraction", "State of charge before and after each physical step"),
        ("quality", "baseline_delivery_error_kw", "kW", "Failure to deliver the accepted energy baseline"),
        ("quality", "signed_tracking_error_kw / absolute_tracking_error_kw", "kW", "FCR requested-versus-delivered tracking error"),
        ("quality", "non_delivery_energy_kwh / saturation_causes", "kWh / labels", "Integrated shortfall and ordered physical constraint causes"),
        ("settlement", "energy revenue / FCR revenue / penalty / total profit", "EUR", "Explicit settlement components"),
        ("audit", "throughput / terminal target / fingerprints", "kWh / values / SHA-256", "Usage, report-only compliance, and reproducibility evidence"),
    ],
    columns=["group", "variable", "unit", "meaning"],
)
display(variable_catalog)


## 3. Optional live ENTSO-E R3 query

This section demonstrates direct retrieval for a caller-selected UTC half-open period. It is **disabled by default** (`RUN_LIVE_QUERY = False`), so ordinary execution remains deterministic and offline. When deliberately enabled, it reads the token from the exported environment or the ignored repository `.env` through the same restricted helper used by the command-line acquisition example. The token is transient: it is never printed, stored in the request object, written to an artifact, or included in a fingerprint.

Edit `LIVE_QUERY_PARAMETERS` to select `requested_start_utc`, `requested_end_utc`, the HTTPS endpoint, and normalization version. The closed France profile then derives two source requests:

| Source response | Derived request profile | Canonical outputs |
| --- | --- | --- |
| A44 | day-ahead `A01`, France as both in/out bidding zone | `day_ahead_price` in EUR/MWh |
| combined A81 | `B95/A52/A01`, France control area; symmetric A03 semantics are validated in the response | `fcr_contracted_volume` in MW and `fcr_capacity_price` in EUR/MW/h |

That is three canonical series from two source responses. The producer queries enclosing Europe/Paris delivery-day windows, validates returned product/unit/coverage semantics, preserves the exact caller period separately, and publishes only an all-series artifact. The cell deliberately uses digest-only raw-response retention and writes beneath ignored artifacts/market/notebook-live-query. A failed query publishes no partial artifact and does not fall back to another source.

> Live results contain restricted real market values. Keep the generated artifact and rendered output private. This exploratory fetch does not replace the reviewed replay fixtures or automatically feed the physical replay; staging a reference fixture remains a separate policy-controlled workflow.


In [ ]:
RUN_LIVE_QUERY = False
LIVE_QUERY_PARAMETERS = {
    "requested_start_utc": "2022-08-18T22:00:00Z",
    "requested_end_utc": "2022-08-19T22:00:00Z",
    "source_endpoint": "https://web-api.tp.entsoe.eu/api",
    "normalization_version": "entsoe-fr-v1",
}
LIVE_ENV_FILE = REPO_ROOT / ".env"
LIVE_DESTINATION = REPO_ROOT / "artifacts/market/notebook-live-query"
live_market = None
live_fetch_result = None

if RUN_LIVE_QUERY:
    from datetime import UTC, datetime, timedelta

    from examples.bess_stacked_services.fetch_french_period import _api_token
    from open_energy_lab.market.entsoe import (
        EntsoeMarketDataRequest,
        build_entsoe_query_windows,
        fetch_entsoe_market_data,
    )

    def _live_utc_parameter(value):
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        if parsed.tzinfo is None or parsed.utcoffset() != timedelta(0):
            raise ValueError("live query boundaries must be explicit UTC instants")
        return parsed.astimezone(UTC)

    live_request = EntsoeMarketDataRequest(
        requested_start_utc=_live_utc_parameter(LIVE_QUERY_PARAMETERS["requested_start_utc"]),
        requested_end_utc=_live_utc_parameter(LIVE_QUERY_PARAMETERS["requested_end_utc"]),
        source_endpoint=LIVE_QUERY_PARAMETERS["source_endpoint"],
        raw_retention_policy="digest_only",
        normalization_version=LIVE_QUERY_PARAMETERS["normalization_version"],
    )
    live_query_windows = build_entsoe_query_windows(live_request)
    display(pd.DataFrame([LIVE_QUERY_PARAMETERS]))
    display(
        pd.DataFrame(
            {"window_id": window.window_id, **dict(window.request_parameters())}
            for window in live_query_windows
        )
    )
    api_token = ""
    try:
        api_token = _api_token(LIVE_ENV_FILE)
        live_fetch_result = fetch_entsoe_market_data(
            live_request,
            api_token=api_token,
            destination_dir=LIVE_DESTINATION,
        )
    finally:
        api_token = ""

    live_market = live_fetch_result.artifact
    display(pd.DataFrame(window.to_dict() for window in live_market.query_windows))
    display(
        pd.DataFrame(
            [
                {
                    "outcome": live_fetch_result.outcome,
                    "artifact_path": str(live_fetch_result.artifact_path),
                    "day_ahead_intervals": len(live_market.series["day_ahead_price"].observations),
                    "fcr_volume_blocks": len(live_market.series["fcr_contracted_volume"].observations),
                    "fcr_price_blocks": len(live_market.series["fcr_capacity_price"].observations),
                    "content_fingerprint": live_market.content_fingerprint,
                }
            ]
        )
    )
else:
    print("Live ENTSO-E query skipped; set RUN_LIVE_QUERY=True to opt in.")


In [ ]:
if live_market is not None:
    live_day_ahead = market_frame(live_market, "day_ahead_price", "price_eur_per_mwh")
    live_fcr_volume = market_frame(live_market, "fcr_contracted_volume", "contracted_volume_mw")
    live_fcr_price = market_frame(live_market, "fcr_capacity_price", "capacity_price_eur_per_mw_h")

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    axes[0].step(*step_data(live_day_ahead, "price_eur_per_mwh"), where="post")
    axes[0].set_ylabel("EUR/MWh")
    axes[0].set_title("Live A44 day-ahead price")
    axes[1].step(*step_data(live_fcr_price, "capacity_price_eur_per_mw_h"), where="post")
    axes[1].set_ylabel("EUR/MW/h")
    axes[1].set_title("Live combined-A81 FCR capacity price")
    axes[2].step(*step_data(live_fcr_volume, "contracted_volume_mw"), where="post")
    axes[2].set_ylabel("MW")
    axes[2].set_xlabel("Europe/Paris delivery time")
    axes[2].set_title("Live combined-A81 contracted FCR volume")
    finish_figure(fig, "00-live-market-query.png")
else:
    print("No live market artifact to plot.")


## 4. Market variables: day-ahead price and FCR market conditions

The upper panel is the hourly A44 price that drives the deterministic energy request; dashed lines are the charge/discharge thresholds. The other panels show the combined-A81 FCR capacity remuneration and total contracted market volume. The replay commits a fixed 5 kW battery share rather than the full system volume.


In [ ]:
threshold = expected["baseline_derivation"]["price_threshold"]
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].step(*step_data(day_ahead, "price_eur_per_mwh"), where="post", color="tab:blue", label="Day-ahead price")
axes[0].axhline(threshold["charge_below_eur_per_mwh"], color="tab:green", linestyle="--", label="Charge threshold")
axes[0].axhline(threshold["discharge_above_eur_per_mwh"], color="tab:red", linestyle="--", label="Discharge threshold")
axes[0].set_ylabel("EUR/MWh")
axes[0].set_title("Day-ahead energy price and controller thresholds")
axes[0].legend(ncol=3)

axes[1].step(*step_data(fcr_price, "capacity_price_eur_per_mw_h"), where="post")
axes[1].set_ylabel("EUR/MW/h")
axes[1].set_title("Symmetric FCR capacity remuneration")

axes[2].step(*step_data(fcr_volume, "contracted_volume_mw"), where="post")
axes[2].set_ylabel("MW")
axes[2].set_xlabel("Europe/Paris delivery time")
axes[2].set_title("Total symmetric FCR contracted volume")
finish_figure(fig, "01-market-variables.png")


## 5. Control variables: threshold request, reserve headroom, and fixed baseline

For each hour, the controller first creates an unmodified `+5/0/-5 kW` threshold request. The battery then reserves symmetric power headroom for the accepted FCR commitment. The shaded interval is the admissible baseline range after headroom reservation; the fixed baseline must remain inside it. This feature validates deterministic schedules but does not optimize energy or reserve bids.


In [ ]:
x = decisions["start_local"].tolist() + [decisions["end_local"].iloc[-1]]
lower = decisions["headroom_lower_kw"].tolist() + [decisions["headroom_lower_kw"].iloc[-1]]
upper = decisions["headroom_upper_kw"].tolist() + [decisions["headroom_upper_kw"].iloc[-1]]
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].fill_between(x, lower, upper, step="post", alpha=0.18, label="Admissible FCR-headroom interval")
axes[0].step(*step_data(decisions, "unmodified_threshold_request_kw"), where="post", label="Unmodified threshold request")
axes[0].step(*step_data(decisions, "fixed_baseline_power_kw"), where="post", linestyle="--", label="Accepted fixed baseline")
axes[0].axhline(expected["battery"]["max_charge_kw"], color="black", linewidth=0.8, linestyle=":")
axes[0].axhline(-expected["battery"]["max_discharge_kw"], color="black", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("Battery power (kW)")
axes[0].set_title("Energy controller and reserved symmetric power headroom")
axes[0].legend(loc="upper center", ncol=3)

axes[1].step(*step_data(fcr_schedule, "capacity_kw"), where="post", label="Committed FCR capacity")
axes[1].step(*step_data(decisions, "committed_fcr_capacity_kw"), where="post", linestyle="--", label="Capacity used in headroom check")
axes[1].set_ylim(0.0, 1.2 * fcr_schedule["capacity_kw"].max())
axes[1].set_ylabel("kW")
axes[1].set_xlabel("Europe/Paris delivery time")
axes[1].set_title("Fixed four-hour symmetric FCR commitment")
axes[1].legend()
finish_figure(fig, "02-control-and-headroom.png")


## 6. Physical variables: activation and requested versus delivered power

The normalized activation command scales the FCR commitment. Requested combined power is `baseline + requested FCR power`; delivered combined power is the battery-feasible result after power, SOC, and efficiency constraints. Delivered FCR power is always measured relative to the accepted baseline, not relative to zero. All 8,640 ten-second points are plotted.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)

axes[0].plot(sample_frame["start_local"], sample_frame["activation"], linewidth=0.8, label="Normalized activation")
axes[0].plot(sample_frame["start_local"], sample_frame["fcr_capacity_kw"] / sample_frame["fcr_capacity_kw"].max(), linewidth=1.0, alpha=0.7, label="FCR capacity (normalized for scale)")
axes[0].axhline(0.0, color="black", linewidth=0.7)
axes[0].set_ylim(-1.05, 1.05)
axes[0].set_ylabel("Normalized command")
axes[0].set_title("FCR activation and available committed capacity")
axes[0].legend()

axes[1].plot(sample_frame["start_local"], sample_frame["baseline_power_kw"], linewidth=1.1, label="Energy baseline")
axes[1].plot(sample_frame["start_local"], sample_frame["requested_combined_power_kw"], linewidth=0.8, label="Requested combined power")
axes[1].plot(sample_frame["start_local"], sample_frame["delivered_combined_power_kw"], linewidth=0.8, linestyle="--", label="Delivered combined power")
axes[1].axhline(0.0, color="black", linewidth=0.7)
axes[1].set_ylabel("kW (+ charge)")
axes[1].set_title("Energy baseline and total battery power")
axes[1].legend(ncol=3)

axes[2].plot(sample_frame["start_local"], sample_frame["requested_fcr_power_kw"], linewidth=0.8, label="Requested FCR power")
axes[2].plot(sample_frame["start_local"], sample_frame["delivered_fcr_power_kw"], linewidth=0.8, linestyle="--", label="Delivered FCR power")
axes[2].axhline(0.0, color="black", linewidth=0.7)
axes[2].set_ylabel("kW (+ charge)")
axes[2].set_xlabel("Europe/Paris delivery time")
axes[2].set_title("Requested and delivered response attributable to FCR")
axes[2].legend()
finish_figure(fig, "03-activation-and-power.png")


## 7. Battery variables: stored energy, SOC, interval energy, and throughput

State is reported at 15-minute checkpoints even though physics advances every 10 seconds. Charge and discharge efficiencies affect stored energy asymmetrically. Throughput is terminal-side charge energy plus discharge energy and is accumulated independently of revenue. The terminal target is report-only: it does not alter dispatch.


In [ ]:
battery = expected["battery"]
target = expected["terminal_soc_target"]
sample_frame["cumulative_throughput_kwh"] = (
    sample_frame["charge_energy_kwh"] + sample_frame["discharge_energy_kwh"]
).cumsum()
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

axes[0].plot(checkpoint_frame["timestamp_local"], checkpoint_frame["stored_energy_kwh"], label="Stored energy")
axes[0].axhline(battery["capacity_kwh"] * battery["min_soc"], linestyle="--", label="Minimum")
axes[0].axhline(battery["capacity_kwh"] * battery["max_soc"], linestyle="--", label="Maximum")
axes[0].axhline(target["target_value"], linestyle=":", color="black", label="Report-only terminal target")
axes[0].set_ylabel("kWh")
axes[0].set_title("Stored battery energy and operating bounds")
axes[0].legend(ncol=4)

axes[1].plot(checkpoint_frame["timestamp_local"], 100 * checkpoint_frame["soc"], label="State of charge")
axes[1].axhline(100 * battery["min_soc"], linestyle="--")
axes[1].axhline(100 * battery["max_soc"], linestyle="--")
axes[1].set_ylabel("SOC (%)")
axes[1].set_title("Post-step state of charge at 15-minute checkpoints")

axes[2].step(*step_data(energy_summary, "charge_energy_kwh"), where="post", label="Charge energy")
discharge_x, discharge_y = step_data(energy_summary, "discharge_energy_kwh")
axes[2].step(discharge_x, [-value for value in discharge_y], where="post", label="Discharge energy (shown negative)")
axes[2].step(*step_data(energy_summary, "net_terminal_energy_kwh"), where="post", linestyle="--", label="Net terminal energy")
axes[2].axhline(0.0, color="black", linewidth=0.7)
axes[2].set_ylabel("kWh per hour")
axes[2].set_title("Charge, discharge, and net terminal energy")
axes[2].legend(ncol=3)

axes[3].plot(sample_frame["start_local"], sample_frame["cumulative_throughput_kwh"], label="Cumulative throughput")
axes[3].set_ylabel("kWh")
axes[3].set_xlabel("Europe/Paris delivery time")
axes[3].set_title("Cumulative terminal-side charge plus discharge energy")
axes[3].legend()
finish_figure(fig, "04-battery-state-and-throughput.png")

display(
    energy_summary[["start_local", "charge_energy_kwh", "discharge_energy_kwh", "net_terminal_energy_kwh", "non_delivery_energy_kwh"]]
)


## 8. Delivery-quality variables: tracking, non-delivery, and saturation

Baseline delivery error identifies failure to deliver the accepted energy baseline. FCR tracking error compares requested and delivered FCR response. Non-delivery integrates only the positive shortfall in response. Saturation events aggregate contiguous power/SOC constraints with an ordered cause, duration, maximum error, and undelivered energy. For visualization only, values below the recorded `1e-6` absolute tolerance are shown as zero; the raw values remain available in `sample_frame` and the tracking summary. Flat zero traces are meaningful: they demonstrate that this reference commitment stays feasible.


In [ ]:
quality_tolerance = expected["expected"]["independent_checks"]["absolute_tolerance"]
quality_columns = ["baseline_delivery_error_kw", "signed_tracking_error_kw", "absolute_tracking_error_kw", "non_delivery_energy_kwh"]
quality_view = sample_frame[["start_local", *quality_columns]].copy()
for column in quality_columns:
    quality_view[column] = quality_view[column].where(quality_view[column].abs() >= quality_tolerance, 0.0)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(quality_view["start_local"], quality_view["baseline_delivery_error_kw"], label="Baseline delivery error")
axes[0].plot(quality_view["start_local"], quality_view["signed_tracking_error_kw"], label="Signed FCR tracking error")
axes[0].plot(quality_view["start_local"], quality_view["absolute_tracking_error_kw"], linestyle="--", label="Absolute FCR tracking error")
axes[0].set_ylim(-quality_tolerance, quality_tolerance)
axes[0].set_ylabel("kW")
axes[0].set_title("Power-delivery error audit")
axes[0].legend(ncol=3)

axes[1].plot(quality_view["start_local"], quality_view["non_delivery_energy_kwh"].cumsum(), label="Cumulative FCR non-delivery")
axes[1].set_ylim(-0.1 * quality_tolerance, quality_tolerance)
axes[1].set_ylabel("kWh")
axes[1].set_xlabel("Europe/Paris delivery time")
axes[1].set_title("Cumulative response non-delivery")
axes[1].legend()
finish_figure(fig, "05-delivery-quality.png")

tracking_table = pd.DataFrame([dict(result.tracking_summary)])
saturation_table = pd.DataFrame(item.to_dict() for item in result.saturation_events)
display(tracking_table)
if saturation_table.empty:
    print("No saturation events: every requested combined/FCR power point was feasible.")
else:
    display(saturation_table)


## 9. Settlement variables: interval revenue, penalties, and total profit

Day-ahead revenue settles the accepted hourly baseline at the A44 price; physical activation does not rewrite that schedule. FCR capacity revenue settles committed kW converted to MW at the A81 EUR/MW/h rate and block duration. Penalty is exactly zero because no deterministic penalty rule is configured. Activation-energy payment, imbalance settlement, and degradation cost are explicitly `not_modeled`, not silently assumed to be zero.


In [ ]:
components = pd.Series(
    {
        "Day-ahead energy": result.settlement.day_ahead_energy_revenue_eur,
        "FCR capacity": result.settlement.fcr_capacity_revenue_eur,
        "Penalty": -result.settlement.penalty_eur,
        "Total profit": result.settlement.total_profit_eur,
    },
    name="EUR",
)
fig, axes = plt.subplots(3, 1, figsize=(14, 11))
axes[0].bar(energy_settlement["start_local"], energy_settlement["revenue_eur"], width=0.035)
axes[0].axhline(0.0, color="black", linewidth=0.7)
axes[0].set_ylabel("EUR")
axes[0].set_title("Hourly day-ahead schedule revenue")

axes[1].bar(fcr_settlement["start_local"], fcr_settlement["revenue_eur"], width=0.14)
axes[1].set_ylabel("EUR")
axes[1].set_title("Four-hour FCR capacity revenue")

components.plot.bar(ax=axes[2], rot=0)
axes[2].axhline(0.0, color="black", linewidth=0.7)
axes[2].set_ylabel("EUR")
axes[2].set_title("Settlement component summary")
finish_figure(fig, "06-settlement.png")

settlement_summary = pd.DataFrame(
    [
        {
            "day_ahead_energy_revenue_eur": result.settlement.day_ahead_energy_revenue_eur,
            "fcr_capacity_revenue_eur": result.settlement.fcr_capacity_revenue_eur,
            "penalty_eur": result.settlement.penalty_eur,
            "throughput_kwh": result.throughput_kwh,
            "total_profit_eur": result.settlement.total_profit_eur,
            "activation_energy": result.settlement.activation_energy_settlement_status,
            "imbalance": result.settlement.imbalance_settlement_status,
            "degradation": result.settlement.degradation_cost_status,
        }
    ]
)
display(settlement_summary)
display(energy_settlement[["start_local", "accepted_power_kw", "price_eur_per_mwh", "revenue_eur"]])
display(fcr_settlement[["start_local", "committed_capacity_kw", "capacity_price_eur_per_mw_per_h", "revenue_eur"]])


## 10. Terminal target and reproducibility audit

The terminal target is a compliance report only. The fingerprint table proves that the loaded inputs, reconstructed schedules, physical trace, and final result match the reviewed reference artifact. Any changed fixture, schedule, simulation semantics, or result fails before the notebook presents the run as reproduced.


In [ ]:
recorded_fingerprints = expected["expected"]["fingerprints"]
actual_fingerprints = {
    "market_artifact": market.artifact_fingerprint,
    "activation_artifact": activation.artifact_fingerprint,
    "energy_schedule": result.input_fingerprints["energy_schedule"],
    "fcr_schedule": result.input_fingerprints["fcr_schedule"],
    "trace": result.trace_fingerprint,
    "result": result.result_fingerprint,
}
fingerprint_audit = pd.DataFrame(
    [
        {"component": name, "recorded": recorded_fingerprints[name], "actual": value, "matches": recorded_fingerprints[name] == value}
        for name, value in actual_fingerprints.items()
    ]
)
display(fingerprint_audit)
display(pd.DataFrame([result.terminal_soc_report.to_dict()]))
assert fingerprint_audit["matches"].all()
assert result.terminal_soc_report is not None
print("Replay validation status:", result.validation_status)
print("Simulation semantics:", result.simulation_semantics_version)


## 11. How to interpret and extend the example

What is working now:

- explicit ENTSO-E R3 acquisition and immutable refresh outside this notebook;
- canonical, source-independent market and activation artifacts;
- deterministic price-threshold, regulation-only, and stacked baseline builders;
- exact multi-rate physical replay with power/SOC feasibility and saturation attribution;
- day-ahead plus FCR-capacity settlement, throughput, terminal reporting, and fingerprints; and
- fast, zero-network offline reproduction from restricted fixtures.

Important limits:

- this is one historical French day and one BESS, not a forecast or profitability guarantee;
- schedules are fixed or deterministic heuristics, not optimized bids;
- FCR activation-energy, imbalance, degradation, aFRR/mFRR, network constraints, and rolling/intraday control are not modeled;
- the zero-penalty result reflects the configured settlement rule, not a universal market rule; and
- private fixture access does not permit public redistribution of values or rendered charts.

To experiment safely, change battery or schedule inputs in a separate local copy and rerun the public simulator APIs. Do not overwrite the reviewed fixtures or present changed runs as matching their fingerprints. A later optimization feature can consume the same artifact, schedule, and replay contracts without moving bidding logic into the market producer or simulator.
